# API 模式
`Starlette` 支持生成 API 模式，例如广泛使用的`OpenAPI` 规范。（以前称为“Swagger”。）

模式生成的工作方式是，通过检查应用程序上的路由 `app.routes`，并使用端点上的文档字符串或其他属性来确定完整的` API 模式`。

`Starlette` 不依赖于任何特定的模式生成或验证工具，但包含一个基于文档字符串生成 `OpenAPI` 模式的简单实现。



In [ ]:
from starlette.applications import Starlette
from starlette.routing import Route
from starlette.schemas import SchemaGenerator


schemas = SchemaGenerator(
    {"openapi": "3.0.0", "info": {"title": "Example API", "version": "1.0"}}
)

def list_users(request):
    """
    responses:
      200:
        description: A list of users.
        examples:
          [{"username": "tom"}, {"username": "lucy"}]
    """
    raise NotImplementedError()


def create_user(request):
    """
    responses:
      200:
        description: A user.
        examples:
          {"username": "tom"}
    """
    raise NotImplementedError()


def openapi_schema(request):
    return schemas.OpenAPIResponse(request=request)


routes = [
    Route("/users", endpoint=list_users, methods=["GET"]),
    Route("/users", endpoint=create_user, methods=["POST"]),
    Route("/schema", endpoint=openapi_schema, include_in_schema=False)
]

app = Starlette(routes=routes)

我们现在可以在“/schema”端点访问 OpenAPI 模式。

您可以使用以下命令直接生成 API `Schema.get_schema(routes)`：

In [ ]:
schema = schemas.get_schema(routes=app.routes)
assert schema == {
    "openapi": "3.0.0",
    "info": {"title": "Example API", "version": "1.0"},
    "paths": {
        "/users": {
            "get": {
                "responses": {
                    200: {
                        "description": "A list of users.",
                        "examples": [{"username": "tom"}, {"username": "lucy"}],
                    }
                }
            },
            "post": {
                "responses": {
                    200: {"description": "A user.", "examples": {"username": "tom"}}
                }
            },
        },
    },
}

您可能还希望能够打印出 API 模式，以便可以使用生成 API 文档等工具。

In [ ]:
if __name__ == '__main__':
    assert sys.argv[-1] in ("run", "schema"), "Usage: example.py [run|schema]"

    if sys.argv[-1] == "run":
        uvicorn.run("example:app", host='0.0.0.0', port=8000)
    elif sys.argv[-1] == "schema":
        schema = schemas.get_schema(routes=app.routes)
        print(yaml.dump(schema, default_flow_style=False))

## 第三方软件包
- [starlette-apispec](https://github.com/Woile/starlette-apispec)
- Starlette 的 APISpec 轻松集成，支持一些对象序列化库。